In [1]:
!pip install ultralytics torch gradio transformers gradio_client google_images_search


In [9]:
!pip install bing-image-downloader

In [5]:
!curl  https://huggingface.co/spaces/fancyfeast/joycaption-watermark-detection/resolve/main/yolo11x-train28-best.pt -L -o yolo11x-train28-best.pt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1331  100  1331    0     0   8631      0 --:--:-- --:--:-- --:--:--  8642
100  109M  100  109M    0     0  50.5M      0  0:00:02  0:00:02 --:--:-- 69.0M


In [11]:
!curl https://huggingface.co/spaces/fancyfeast/joycaption-watermark-detection/resolve/main/far5y1y5-8000.pt?download=true -L -o far5y1y5-8000.pt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1323  100  1323    0     0   8393      0 --:--:-- --:--:-- --:--:--  8426
100  341M  100  341M    0     0   161M      0  0:00:02  0:00:02 --:--:--  191M


In [10]:
from huggingface_hub import notebook_login
notebook_login()

In [11]:
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TVF
from transformers import Owlv2VisionModel
from torch import nn
import torch
import torch.nn.functional as F
import time


# OWLv2 classification head
class DetectorModelOwl(nn.Module):
	owl: Owlv2VisionModel

	def __init__(self, model_path: str, dropout: float, n_hidden: int = 768):
		super().__init__()

		owl = Owlv2VisionModel.from_pretrained(model_path)
		assert isinstance(owl, Owlv2VisionModel)
		self.owl = owl
		self.owl.requires_grad_(False)
		self.transforms = None

		self.dropout1 = nn.Dropout(dropout)
		self.ln1 = nn.LayerNorm(n_hidden, eps=1e-5)
		self.linear1 = nn.Linear(n_hidden, n_hidden * 2)
		self.act1 = nn.GELU()
		self.dropout2 = nn.Dropout(dropout)
		self.ln2 = nn.LayerNorm(n_hidden * 2, eps=1e-5)
		self.linear2 = nn.Linear(n_hidden * 2, 2)

	def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None):
		with torch.autocast("cuda", dtype=torch.bfloat16):
			# Embed the image
			outputs = self.owl(pixel_values=pixel_values, output_hidden_states=True)
			x = outputs.last_hidden_state  # B, N, C

			# Linear
			x = self.dropout1(x)
			x = self.ln1(x)
			x = self.linear1(x)
			x = self.act1(x)

			# Norm and Mean
			x = self.dropout2(x)
			#x = x.mean(dim=1)
			x, _ = x.max(dim=1)
			x = self.ln2(x)

			# Linear
			x = self.linear2(x)

		if labels is not None:
			loss = F.cross_entropy(x, labels)
			return (x, loss)

		return (x,)


def owl_predict(image: Image.Image) -> bool:
	# Process the image
	# Pad to square
	big_side = max(image.size)
	new_image = Image.new("RGB", (big_side, big_side), (128, 128, 128))
	new_image.paste(image, (0, 0))

	# Resize to 960x960
	preped = new_image.resize((960, 960), Image.BICUBIC)  # Bicubic performed best in my tests (even compared to Lanczos)
	#preped = new_image.resize((1008, 1008), Image.BICUBIC)  # Bicubic performed best in my tests (even compared to Lanczos)

	# Convert to tensor and normalize
	preped = TVF.pil_to_tensor(preped)
	preped = preped / 255.0
	input_image = TVF.normalize(preped, [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])

	# Run
	logits, = model(input_image.to('cpu').unsqueeze(0), None)
	probs = F.softmax(logits, dim=1)
	prediction = torch.argmax(probs.cpu(), dim=1)

	return prediction.item() == 1


def predict(image: Image.Image, conf_threshold: float):
	# OWLv2
	owl_prediction = owl_predict(image)
	return owl_prediction


# Load OWLv2 classification model
model = DetectorModelOwl("google/owlv2-base-patch16-ensemble", dropout=0.0)
model.load_state_dict(torch.load("/content/far5y1y5-8000.pt", map_location="cpu"))
model.eval()


DetectorModelOwl(
  (owl): Owlv2VisionModel(
    (vision_model): Owlv2VisionTransformer(
      (embeddings): Owlv2VisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
        (position_embedding): Embedding(3601, 768)
      )
      (pre_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (encoder): Owlv2Encoder(
        (layers): ModuleList(
          (0-11): 12 x Owlv2EncoderLayer(
            (self_attn): Owlv2Attention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Owlv2MLP(
              (activation_fn): QuickGELUActivation()
        

In [23]:
start = time.monotonic_ns()
water = Image.open("/content/waterMarked.webp")
nonWater = Image.open("/content/nonWaterMarked.jpg")
print(predict(water,0.5))
print(predict(nonWater,0.5))
print((time.monotonic_ns()-start)/10**9)


True
False
49.333360999


In [14]:
from icrawler.builtin import BaiduImageCrawler, BingImageCrawler, GoogleImageCrawler

bing_crawler= BingImageCrawler(storage={'root_dir': '/content/searchedImage'})
# downloader.download('بویلر آب جوش 10 لیتری برقی مشکی', limit=10, output_dir='/content/searchedImage',filter="photo", adult_filter_off=True, force_replace=False, timeout=60, verbose=True)

# google_crawler = GoogleImageCrawler(storage={'root_dir': '/content/searchedImage'})
bing_crawler.crawl(keyword='بویلر آب جوش 10 لیتری برقی مشکی', max_num=10)


ERROR:downloader:Exception caught when downloading file https://tajhizsho.com/wp-content/uploads/2024/01/8-2.jpg, error: HTTPSConnectionPool(host='tajhizsho.com', port=443): Max retries exceeded with url: /wp-content/uploads/2024/01/8-2.jpg (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7cec4686ea20>, 'Connection to tajhizsho.com timed out. (connect timeout=5)')), remaining retry times: 2
ERROR:downloader:Exception caught when downloading file https://tajhizsho.com/wp-content/uploads/2024/01/8-2.jpg, error: HTTPSConnectionPool(host='tajhizsho.com', port=443): Max retries exceeded with url: /wp-content/uploads/2024/01/8-2.jpg (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7cec4686f080>, 'Connection to tajhizsho.com timed out. (connect timeout=5)')), remaining retry times: 1
ERROR:downloader:Exception caught when downloading file https://tajhizsho.com/wp-content/uploads/2024/01/8-2.jpg, error: HTTPSConnectionPool(host='tajhi

KeyboardInterrupt: 

In [15]:
! curl https://tajhizsho.com/wp-content/uploads/2024/01/8-2.jpg -L -o img.jpg


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0

ERROR:downloader:Exception caught when downloading file https://industrial-kitchen.com/wp-content/uploads/2018/05/boiler3.jpg, error: HTTPSConnectionPool(host='industrial-kitchen.com', port=443): Max retries exceeded with url: /wp-content/uploads/2018/05/boiler3.jpg (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7cec47061190>, 'Connection to industrial-kitchen.com timed out. (connect timeout=5)')), remaining retry times: 0


  0     0    0     0    0     0      0      0 --:--:--  0:01:21 --:--:--     0^C
